In [3]:
import os
import sys
from contextlib import contextmanager, redirect_stdout
from typing import Generator

import matplotlib.pyplot as plt
import numpy as np
import simpy
from matplotlib.axes import Axes
from matplotlib.figure import Figure

RNG = np.random.default_rng(seed=0)

## Closed Queuing network simulation

In [ ]:
class ClosedNetworkSim:
    def __init__(self, N=40, sim_time =20000, seed = None):
        self.env = simpy.Environment() # represents simulation environment, processes(functions that yield events) run inside it
        # self.beta_s_cpu = 1. /lambda_s_cpu # processing times
        # self.beta_s_disk = 1. /lambda_s_disk
        self.rng = random.Random(seed)

       # Resources (needed for sure)
        self.cpu = simpy.Resource(self.env, capacity=1) # create cpu, capacity = how many processes can use it the same it 
        self.disk_slow = simpy.Resource(self.env, capacity=1)
        self.disk_fast = simpy.Resource(self.env, capacity=1)

        # parameters
        self.fast_speed = 1000
        self.slow_speed = 100
        self.cpu_time = 2
        self.disk_cycles = 3000
        self.rest = 15

        # To check stats
        self.cycles_done = 0
        self.cycles_duree = []

    # Suggested structure (to discuss)
    def cpu_service(self):
        return self.rng.expovariate(1/self.cpu_time)
        # a chaque fois que un job arrive au cpu, on veut savoir combien de temps ça va prendre. 
        # On sait que la moyenne est de 2 sec et qu'il est explonentiel
        # a chaque fois on se tire un nombre au hasard de durée de cpu de exp(2)

    def rest_time(self):
        return self.rng.expovariate(1/self.rest_time)

    def disk_time_fast(self):
        return self.disk_cycles/self.fast_speed # nums statiques, pas sure que on en a besoin au final

    def disk_time_slow(self):
        return self.disk_cycles/self.slow_speed 


    def choose_disk(self):
        pass # J'en sais rien lol à voir

    def job(self, id_job):
        start_cycle_time ( self.env.now)
        
        while True:
            # requete cpu, service cpu
            with self.cpu.request() as request:
                yield request
                yield self.env.timeout(self.cpu_service())
            
            # choix disk, requete, service 
            chosen_disk = self.choose_disk()
            with chosen_disk.request() as d_request:
                yield d_request
                yield self.env.timeout(chosen_disk) # a revoir bancal

            # repost
            yield self.env.timeout(self.rest_time)

            cycle = self.env.now - start_cycle_time # durée du cycle

            self.cycle_times.append(cycle) # ajoutter à la liste des durées des cycles
            start_cycle_time = self.env.now 
            self.completed_cycles += 1 # incrementer compteur


    def run(self, t: float) -> None:
        for i in range(self.N):
            self.env.process(self.job(i)) # self.job est un process qui vit dans l'env, commence timer
            self.env.run(until=self.sim_time) # avance temps simulé, jusqu'au temps indiqué dans sim time

In [ ]:
simple_closed_net = SimpleClosedNet(0.2)
simple_closed_net.run(10)

#### (3p) What will be the maximal system throughput in terms of number of jobs, given two different load balancing strategies for sending jobs between the fast and slow disk? You need to come up with two load balancing strategies and compare them.

#### (3p) What will the system throughput and average response time of a job be if a faster CPU is used? Say, CPU time is reduced to 1 seconds on average and the rest of the system remains the same.

#### (3p) What will the system throughput and average response time of a job be if a second fast disk is added? Again, you need to compare the throughput under two different load balancing strategies.

#### (3p) What will the system throughput and average response time of a job be if a faster CPU is used and a second fast disk is added? And, you use the better load balancing strategies out of two you propose in the first question?

#### (5pt) If you can answer all the above questions with different number of N and plot them in the following style:



## Open Queuing Network simulation

#### (4pt) Use your simulator to find the maximum sustainable throughput of such a system in terms of number of jobs. Support your answer with simulation results.

#### (4pt) Find out the average response times for following cases over three arrival rates of your choice: case (i) - a single queue in front of fast and slow disk; case (ii) - a separate queue for each disk, jointly applying the shortest queue load balancing strategy when sending the jobs from the CPU to disk.